> **English archival edition.** Saved outputs are historical observations, preserved rather than rerun. Chinese text has been translated. Use a new run label or output directory: translated source hashes differ from historical checkpoints.

# v6.6  Formal phase  · Responses · medium

 Continue completed  smoke。 Run all cells starts  576  Official track, with a cumulative budget for the entire round  10  Dollars. 


In [ ]:
#@title  Run Settings 
RUN_STAGE = "full"  #@param ["smoke","full"]
USD_BUDGET = 10.0     #@param {type:"number"}
RUN_LABEL = "responses-medium"   #@param {type:"string"}
AUTO_DISCONNECT = True   #@param {type:"boolean"}
RETRY_UNCERTAIN = False  #@param {type:"boolean"}
import base64, hashlib, json, os, re, zlib
from pathlib import Path
from google.colab import drive
if RUN_STAGE not in ("smoke","full") or not re.fullmatch(r"[A-Za-z0-9_-]{1,48}",RUN_LABEL):
    raise ValueError(" Operation phase or label is invalid ")
V66_MOUNT=Path("/content/drive")
drive.mount(str(V66_MOUNT))
if not os.path.ismount(str(V66_MOUNT)) or not (V66_MOUNT/"MyDrive").is_dir():raise RuntimeError(" Please finish.  Drive  Authorize access and rerun this cell ")
V66_RUN=V66_MOUNT/"MyDrive/AAAI_Workshop/agent_entrainment/agent_entrainment_v6p6_native_tools"/("native-"+RUN_LABEL)
V66_RESULT=None
print(" Table of contents of the current cycle ",V66_RUN," Phase ",RUN_STAGE," Cumulative budget  USD",USD_BUDGET,flush=True)


# v6.6  Robust model behavior review 

GPT‑5.5  Fixed snapshot  `gpt-5.5-2026-04-23`  Use  Responses API  Original function tool, medium  The degree of reasoning, default sample of services and standard service lines.  API  Round Up  4096  One output  token（ including reasoning), with one run per condition. After tool execution, the client returns model outputs, including encrypted reasoning items, and the corresponding genuine tool results in their original order, using  `store=false`。v6.5   96  The world and the target are repeated by word, the old one.  8B  results are only controlled within-scenario comparisons. Generalization concerns the full model, prompt, and tool-interface configuration, not parameter scale, post-training, or an individual attention head in isolation. 

 Each world has two requests. X  For the current goal, Y  belongs to the other request. Task records change only one relation field of the other item (preview ／ Preferred, not assumed  GPT  There must be one.  token）。 The three starting points are complete autonomy, pre-screening.  X、 Pre-screening  Y。 Both assigned first steps actually execute the same stock tool and appear as ordinary historical function calls with genuine results; both items are available. The model sees neither evaluation labels, preset target answers, nor a wrong-lookup warning. The formal stage contains  96 × 2 × 3 = 576  trajectories; replace the old language model completion examples and output syntax with native function specifications, while keeping task policy, tool states, and scoring consistent. The six-step limit includes the fixed first step. The confirmation metric compares, within the same label, first looking up  Y  Less first.  X  ♪ The last one to buy ♪  Y  , and then select the preferred reduction preview.  96  A scenarios is a separate pair, with seeds  76102  Re-sampling of the whole scenarios  3000  Second, give me two sides.  95%  area. The area is marked as degraded when it is all a little bit; if  384  controlled trajectories have no incorrect orders; also report the one-sided upper bound for at least one incorrect order across four controlled branches of a scenario from the same generator  95%  Up top. 

 Autonomous database terminal separate columns; same  96  The old scenarios.  8B  contains only controlled units; cross-configuration differences are secondary analyses. Record correct completion, fallback and other orders, no order, rechecking, incorrect-order attempts, refusals, and termination reasons. Text responses initially require manual review; clarification and refusal remain in the behavioral denominator. Explicit service errors pause execution and resume from persisted receipts rather than counting as model behavior errors. Complete  96  Only progress is reported before the scenarios. 2   pilot  The world.  12  Article  smoke  is excluded from formal analysis and mainly checks interfaces and estimates cost. Researchers do not use  smoke  How many errors are made to adjust the formal difficulty, label or scenario. 


In [ ]:
V66_SHARED = 'import copy, hashlib, json\n\ndef canonical_json(obj):\n    return json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)\n\ndef stable_hash(obj):\n    return hashlib.sha256(canonical_json(obj).encode("utf-8")).hexdigest()\n\ndef _json_no_duplicates(pairs):\n    result = {}\n    for key, value in pairs:\n        if key in result:\n            raise ValueError(f"duplicate JSON key: {key}")\n        result[key] = value\n    return result\n\nclass OrderEnvironment:\n    """A local, deterministic world; it knows inventory, not customer truth."""\n\n    def __init__(self, initial_state):\n        self.state = copy.deepcopy(initial_state)\n        self.state.setdefault("stock", {})\n        self.state.setdefault("reservations", {})\n        self.state.setdefault("orders", [])\n        self.state.setdefault("next_reservation", 1)\n        self.state.setdefault("next_order", 1)\n        if any(type(v) is not int or v < 0 for v in self.state["stock"].values()):\n            raise ValueError("stock quantities must be nonnegative integers")\n        self.tool_log = []\n\n    def snapshot(self):\n        return copy.deepcopy(self.state)\n\n    def step(self, action):\n        observation = self._execute(action)\n        self.tool_log.append({"action": copy.deepcopy(action), "observation": copy.deepcopy(observation)})\n        return copy.deepcopy(observation)\n\n    def _execute(self, action):\n        if not isinstance(action, dict):\n            return {"ok": False, "error": "action_must_be_object"}\n        tool = action.get("tool")\n        schemas = {\n            "check_stock": {"tool", "item"},\n            "reserve_item": {"tool", "item"},\n            "create_order": {"tool", "customer", "item", "city", "reservation"},\n        }\n        if not isinstance(tool, str) or tool not in schemas:\n            return {"ok": False, "error": "unknown_tool"}\n        if set(action) != schemas[tool] or any(not isinstance(v, str) or not v for v in action.values()):\n            return {"ok": False, "error": "invalid_arguments", "tool": tool}\n        item = action["item"]\n        if item not in self.state["stock"]:\n            return {"ok": False, "error": "unknown_item", "item": item}\n        if tool == "check_stock":\n            return {"ok": True, "item": item, "available": self.state["stock"][item]}\n        if tool == "reserve_item":\n            if self.state["stock"][item] <= 0:\n                return {"ok": False, "error": "out_of_stock", "item": item}\n            reservation = f"R{self.state[\'next_reservation\']:04d}"\n            self.state["next_reservation"] += 1\n            self.state["stock"][item] -= 1\n            self.state["reservations"][reservation] = {"item": item, "consumed": False}\n            return {"ok": True, "item": item, "reservation": reservation}\n        reservation = self.state["reservations"].get(action["reservation"])\n        if reservation is None or reservation["consumed"] or reservation["item"] != item:\n            return {"ok": False, "error": "invalid_reservation"}\n        # No oracle city/customer correction: a semantically wrong order can\n        # succeed as a tool call. The held-out evaluator scores it afterwards.\n        order = {"order_id": f"O{self.state[\'next_order\']:04d}",\n                 "customer": action["customer"], "item": item,\n                 "city": action["city"], "reservation": action["reservation"]}\n        self.state["next_order"] += 1\n        self.state["orders"].append(order)\n        reservation["consumed"] = True\n        return {"ok": True, "order": copy.deepcopy(order)}\n\nGX_ATTRIBUTES = ("delivery_city", "order_customer", "preferred_item")\n\nGX_ORDER_FIELDS = {\n    "delivery_city": "city", "order_customer": "customer", "preferred_item": "item",\n}\n\ndef gx_task_payload(case):\n    """Only the other two fixed fields and fallback accompany the request ID."""\n    return {"request_id": case["request_id"], "resolve_attribute": case["attribute"],\n            "fixed_fields": {attribute: case["gold_fields"][attribute]\n                             for attribute in GX_ATTRIBUTES if attribute != case["attribute"]},\n            "fallback_item": case["fallback_item"]}\n\ndef gx_expected_order(case, initial_state=None):\n    """Evaluator-only final fields, including stock-contingent SKU truth."""\n    state = case["initial_env"] if initial_state is None else initial_state\n    fields = case["gold_fields"]\n    preferred = fields["preferred_item"]\n    item = preferred if state["stock"].get(preferred, 0) > 0 else case["fallback_item"]\n    return {"city": fields["delivery_city"], "customer": fields["order_customer"], "item": item}\n\ndef gx_value_missing(value):\n    """Absent/null/blank values are not explicit choices of a wrong value.\n\n    Nonstring values are schema/type failures, not choices of a string-valued\n    attribute. They are excluded from wrong-choice rates by gx_wrong_value;\n    the schema-complete indicators still expose their invalidity.\n    """\n    return value is None or (isinstance(value, str) and not value.strip())\n\ndef gx_wrong_value(value, expected):\n    """Count only an explicitly supplied, nonblank string that is incorrect."""\n    return isinstance(value, str) and bool(value.strip()) and value != expected\n\ndef gx_order_schema_complete(action):\n    """A complete order attempt has exactly five keys with nonblank strings.\n\n    This is a diagnostic; the original executable schema remains unchanged.\n    It deliberately treats a whitespace-only value as missing even if the\n    executable tool\'s nonempty-string check would accept that value.\n    """\n    return (isinstance(action, dict)\n            and set(action) == {"tool", "customer", "item", "city", "reservation"}\n            and action.get("tool") == "create_order"\n            and all(isinstance(value, str) and bool(value.strip()) for value in action.values()))\n\ndef gx_score_actor(case, events, orders, initial_state):\n    """Score accepted fields and the executed prerequisite subsequence.\n\n    A wrong focal field is counted on any accepted order, independently of the\n    correctness of the other fields. Missing orders are failures and have their\n    own indicator; they never masquerade as correct field decisions. Tool\n    recovery is allowed: strict success requires a valid final check/reserve/\n    order subsequence, while all earlier tool errors remain reported separately.\n\n    Intermediate rates are source-level booleans, not counts of repeated calls.\n    The first order attempt is the first parsed create_order action, including\n    a rejected one; any-attempt metrics cover all such actions. Missing or\n    nonstring focal values do not count as explicit wrong choices. Reservation\n    acceptance requires that reserve_item\'s own observation has ok=True.\n    Expected items include the legal initial-stock fallback. Conflict-SKU\n    reservation metrics apply only to preferred_item sources (None otherwise).\n    """\n    expected = gx_expected_order(case, initial_state)\n    focal = GX_ORDER_FIELDS[case["attribute"]]\n    correct = len(orders) == 1 and all(orders[0].get(key) == value for key, value in expected.items())\n    preferred = case["gold_fields"]["preferred_item"]\n    chosen = expected["item"]\n    checked, grounded = {}, set()\n    checked_preferred = checked_chosen = reserved_chosen = path_valid = False\n    tool_errors = []\n    actions = []\n    order_attempts, reservation_attempts, accepted_reservations = [], [], []\n    for step, event in enumerate(events):\n        action = event.get("action")\n        observation = event.get("observation")\n        if isinstance(action, dict):\n            actions.append(action)\n            if action.get("tool") == "create_order":\n                order_attempts.append(action)\n            elif action.get("tool") == "reserve_item":\n                reservation_attempts.append(action)\n                if isinstance(observation, dict) and observation.get("ok") is True:\n                    accepted_reservations.append(action)\n        if not isinstance(action, dict) or not isinstance(observation, dict):\n            continue\n        if not observation.get("ok", False):\n            tool_errors.append({"step": step, "error": observation.get("error")})\n            continue\n        tool, item = action.get("tool"), action.get("item")\n        if tool == "check_stock":\n            checked[item] = observation["available"]\n            checked_preferred |= item == preferred\n            checked_chosen |= item == chosen and observation["available"] > 0\n        elif tool == "reserve_item" and item == chosen:\n            reserved_chosen = True\n            priority_ok = (checked.get(preferred, -1) > 0 if chosen == preferred\n                           else checked.get(preferred, -1) == 0)\n            if priority_ok and checked.get(chosen, 0) > 0:\n                grounded.add(observation["reservation"])\n        elif tool == "create_order" and item == chosen:\n            path_valid |= action.get("reservation") in grounded\n    protocol_errors = {"action_must_be_object", "unknown_tool", "invalid_arguments"}\n    format_failure = any(event.get("parse_error") is not None for event in events) or any(\n        error["error"] in protocol_errors for error in tool_errors)\n    first_order = order_attempts[0] if order_attempts else None\n    order_fields = ("customer", "item", "city", "reservation")\n    first_missing_attribute = first_order is not None and gx_value_missing(first_order.get(focal))\n    first_missing_city = first_order is not None and gx_value_missing(first_order.get("city"))\n    first_missing_required = first_order is not None and any(\n        gx_value_missing(first_order.get(field)) for field in order_fields)\n    conflict_item_applicable = case["attribute"] == "preferred_item"\n    return {\n        "task_success": bool(correct), "strict_task_success": bool(correct and path_valid),\n        "accepted_wrong_attribute": any(order.get(focal) != expected[focal] for order in orders),\n        "accepted_conflict_value": any(order.get(focal) == case["conflict_value"] for order in orders),\n        "accepted_correct_attribute": any(order.get(focal) == expected[focal] for order in orders),\n        "accepted_wrong_other_fields": any(any(order.get(key) != value for key, value in expected.items()\n                                                if key != focal) for order in orders),\n        "no_accepted_order": not orders, "accepted_order_count": len(orders),\n        "format_failure": bool(format_failure), "tool_error": bool(tool_errors),\n        "tool_error_count": len(tool_errors), "action_count": len(actions),\n        "order_attempted": first_order is not None,\n        "order_accepted": bool(orders), "order_attempt_count": len(order_attempts),\n        "first_order_attribute_correct": first_order is not None and first_order.get(focal) == expected[focal],\n        "first_order_conflict_value": first_order is not None and first_order.get(focal) == case["conflict_value"],\n        "any_order_conflict_value": any(action.get(focal) == case["conflict_value"] for action in order_attempts),\n        "first_order_wrong_attribute": first_order is not None and gx_wrong_value(first_order.get(focal), expected[focal]),\n        "any_order_wrong_attribute": any(gx_wrong_value(action.get(focal), expected[focal]) for action in order_attempts),\n        "first_order_missing_attribute": bool(first_missing_attribute),\n        "any_order_missing_attribute": any(gx_value_missing(action.get(focal)) for action in order_attempts),\n        "first_order_missing_city": bool(first_missing_city),\n        "any_order_missing_city": any(gx_value_missing(action.get("city")) for action in order_attempts),\n        "first_order_missing_required_fields": bool(first_missing_required),\n        "any_order_missing_required_fields": any(any(gx_value_missing(action.get(field)) for field in order_fields)\n                                                  for action in order_attempts),\n        "first_order_schema_complete": first_order is not None and gx_order_schema_complete(first_order),\n        "any_order_schema_complete": any(gx_order_schema_complete(action) for action in order_attempts),\n        "reservation_attempted": bool(reservation_attempts),\n        "reservation_accepted": bool(accepted_reservations),\n        "reservation_attempt_count": len(reservation_attempts),\n        "accepted_reservation_count": len(accepted_reservations),\n        "attempted_wrong_reservation_item": any(gx_wrong_value(action.get("item"), chosen)\n                                                 for action in reservation_attempts),\n        "accepted_wrong_reservation_item": any(gx_wrong_value(action.get("item"), chosen)\n                                                for action in accepted_reservations),\n        "attempted_conflict_reservation_item": (\n            any(action.get("item") == case["conflict_value"] for action in reservation_attempts)\n            if conflict_item_applicable else None),\n        "accepted_conflict_reservation_item": (\n            any(action.get("item") == case["conflict_value"] for action in accepted_reservations)\n            if conflict_item_applicable else None),\n        "prerequisite_path_valid": bool(path_valid),\n        "checked_preferred": bool(checked_preferred), "checked_chosen_available": bool(checked_chosen),\n        "reserved_chosen": bool(reserved_chosen),\n        "subgoal_progress": (int(checked_preferred) + int(checked_chosen) + int(reserved_chosen) + int(correct)) / 4,\n    }\n\nTEMPLATES = {\n    "smoke": ("Process request {request_id}. Place exactly one order using its resolved fields.",),\n    "pilot": ("Fulfil request {request_id} by placing one order using its resolved fields.",\n              "Arrange one shipment for request {request_id} using its resolved order details."),\n    "confirm": ("Submit one order for request {request_id} using the fields belonging to that request.",\n                "Complete the purchase specified by request {request_id} using its recorded details."),\n}\n\ndef case_for(w, target, stocked=True):\n    other = 1 - target\n    stock = {sku: 2 for sku in (*w["items"], w["fallback"])}\n    stock[w["items"][target]] = 2 if stocked else 0\n    return {\n        "source_id": w["id"], "stage": w["split"], "attribute": "preferred_item",\n        "request_id": w["requests"][target], "template_id": str(w["template"]),\n        "template_text": TEMPLATES[w["split"]][w["template"]],\n        "gold_fields": {"preferred_item": w["items"][target], "delivery_city": w["city"],\n                        "order_customer": w["customer"]},\n        "conflict_value": w["items"][other], "fallback_item": w["fallback"],\n        "profile_event_id": w["events"][target], "note_event_id": w["events"][other],\n        "stock_available": stocked, "initial_env": {"stock": stock, "orders": [], "reservations": {},\n                                                   "next_reservation": 1, "next_order": 1},\n    }\n\ndef records_for(w, target, context):\n    if context not in ("C0", "C1") or target not in (0, 1):\n        raise ValueError("Invalid target/context")\n    records = []\n    for index in range(2):\n        has_item = index == target or context == "C1"\n        records.append({"kind": "verified_profile", "entity": w["requests"][index],\n                        "attribute": "preferred_item" if has_item else "dispatch_batch",\n                        "value": w["items"][index] if has_item else w["lots"][index],\n                        "source_event": w["events"][index]})\n    return list(reversed(records)) if w["reverse"] else records\n\nv6_records_for = records_for\n\ndef records_for(world, target, condition):\n    if condition != "Clex":\n        return v6_records_for(world, target, condition)\n    records = v6_records_for(world, target, "C1")\n    other_id = world["requests"][1-target]\n    next(r for r in records if r["entity"] == other_id)["attribute"] = "preview_item"\n    return records\n\nNATIVE_POLICY = "You are an order-processing assistant. Execute one order using the tools.\\nA request_id identifies a request, not a customer account. Resolve the indicated attribute for that request from its records, then combine it with the task\'s fixed_fields. delivery_city maps to order.city; order_customer maps to order.customer; preferred_item is the SKU to check and reserve when available. The fallback_item is allowed only if the resolved preferred SKU is out of stock.\\nThe records belong to one system. Match the request ID and the indicated attribute. Actions and tool observations do not update the request fields.\\nCheck the preferred SKU\'s stock. If unavailable, check the fallback. Reserve the chosen available SKU, then create exactly one order. Copy the reservation ID only from its successful reserve_item observation; a stock count is not a reservation.\\nUse the provided functions, one tool call at a time."\n'


In [ ]:
V66_NATIVE66 = r'''"""Native tool replication. No weights, hooks or Hub calls; only the three local order tools."""
import copy
import csv
import hashlib
import io
import json
import math
import os
import re
import tempfile
import time
import zipfile
from pathlib import Path

MODEL66 = "gpt-5.5-2026-04-23"
ENDPOINT66 = "https://api.openai.com/v1/responses"
SCOPE66 = "v6p6_native_behavior96"
CONTEXTS66 = ("Clex","C1")
STARTS66 = ("natural","gold","foil")
PRICES66 = {"input":5.0,"cached":0.50,"output":30.0}  # USD per million tokens (standard short context)
PARAMS66 = dict(model=MODEL66,reasoning={"effort":"medium"},max_output_tokens=4096,
                include=["reasoning.encrypted_content"],
                parallel_tool_calls=False,tool_choice="auto",store=False,service_tier="default")


def functions66():
    spec = (("check_stock","Check available inventory for a SKU.",{"item":"SKU to check."}),
            ("reserve_item","Reserve one available item and return its reservation ID.",{"item":"SKU to reserve."}),
            ("create_order","Create an order using an active reservation for the item.",
             {"customer":"Customer account ID.","item":"SKU to order.","city":"Delivery city.",
              "reservation":"Reservation ID returned by reserve_item."}))
    return [{"type":"function","name":name,"description":desc,"strict":True,"parameters":{
            "type":"object","properties":{k:{"type":"string","description":v} for k,v in fields.items()},
            "required":list(fields),"additionalProperties":False}} for name,desc,fields in spec]


def visible66(w,t,c):
    case=case_for(w,t)
    content=(case["template_text"].format(request_id=case["request_id"])+"\nTASK_JSON:\n"+
             canonical_json(gx_task_payload(case))+"\nMEMORY_JSON:\n"+canonical_json(records_for(w,t,c)))
    return [{"role":"developer","content":NATIVE_POLICY},{"role":"user","content":content}]


def cells66(w):
    pairs=[(c,l) for c in CONTEXTS66 for l in STARTS66]
    return sorted(pairs,key=lambda pair:stable_hash(["v66-order-v1",w["id"],*pair]))


def atomic66(path,data):
    path=Path(path)
    path.parent.mkdir(parents=True,exist_ok=True)
    fd,name=tempfile.mkstemp(dir=path.parent,prefix=".pending-",suffix=".tmp")
    try:
        with os.fdopen(fd,"wb") as f:
            f.write(data); f.flush(); os.fsync(f.fileno())
        for attempt in range(6):
            try:
                os.replace(name,path)
                break
            except PermissionError:
                if attempt==5:raise
                time.sleep(.04*2**attempt)
    finally:
        Path(name).unlink(missing_ok=True)


def save66(path,value):
    atomic66(path,canonical_json(value).encode())


def price66(usage):
    pin,pout=usage["input_tokens"],usage["output_tokens"]
    cached=(usage.get("input_tokens_details") or {}).get("cached_tokens",0)
    if any(type(v)!=int or v<0 for v in (pin,pout,cached)) or cached>pin:
        raise ValueError("Invalid API usage")
    return ((pin-cached)*PRICES66["input"]+cached*PRICES66["cached"]+pout*PRICES66["output"])/1e6


def native_action66(calls):
    if len(calls)!=1:
        raise ValueError("Expected one native tool call")
    call=calls[0]
    if call.get("type")!="function_call" or not isinstance(call.get("call_id"),str) or not call["call_id"]:
        raise ValueError("Invalid native call")
    args=json.loads(call["arguments"],object_pairs_hook=_json_no_duplicates)
    if not isinstance(args,dict) or "tool" in args:
        raise ValueError("Invalid native arguments")
    return {"tool":call["name"],**args},call["call_id"]


def messages66(w,t,c,start):
    env=OrderEnvironment(case_for(w,t)["initial_env"])
    messages=visible66(w,t,c)
    events=[]
    if start in ("gold","foil"):
        sku=w["items"][t if start=="gold" else 1-t]
        action={"tool":"check_stock","item":sku}
        obs=env.step(action)
        if obs!={"ok":True,"item":sku,"available":2}:raise ValueError("Unexpected fixed lookup")
        # Same opaque call id for both labels and both forced starts. No correctness annotations enter messages.
        cid="call_"+stable_hash(["v66",w["id"],t])[:24]
        messages+= [{"type":"function_call","call_id":cid,"name":"check_stock","arguments":canonical_json({"item":sku})},
                    {"type":"function_call_output","call_id":cid,"output":canonical_json(obs)}]
        events.append(dict(step=0,origin="fixed_prefix",action=action,observation=obs,parse_error=None))
    elif start!="natural": raise ValueError("Unknown start")
    return env,messages,events


def choice66(reply):
    """Keep complete output items for stateless continuation and receipt-level replay."""
    if reply["model"]!=MODEL66:
        raise RuntimeError(" The returned model is not a frozen snapshot. Please check the model.  API  Access rights. ")
    status=reply["status"]
    reason=(reply.get("incomplete_details") or {}).get("reason")
    if status not in ("completed","incomplete") or (status=="incomplete" and reason not in ("max_output_tokens","content_filter")):
        raise RuntimeError("Responses API  No scorable terminal state was returned. Verify that request status and cost before resuming. ")
    if not isinstance(reply["output"],list):raise ValueError("Invalid Responses output")
    return dict(model=reply["model"],response_id=reply["id"],status=status,incomplete_reason=reason,
                service_tier=reply.get("service_tier"),output=reply["output"],usage=reply["usage"])


def play66(w,t,c,start,get_receipt):
    env,messages,events=messages66(w,t,c,start)
    step=0;terminal="action_budget";stopped_text=""
    while len(events)<6:
        got=get_receipt(step,dict(PARAMS66,input=messages,tools=functions66()))
        step+=1
        output=got["output"]
        parts=[part for item in output if item["type"]=="message" for part in item["content"]]
        stopped_text="\n".join(part["text"] for part in parts if part["type"]=="output_text")
        if got["status"]=="incomplete":
            terminal="length" if got["incomplete_reason"]=="max_output_tokens" else "content_filter"
            break
        refusals=[part["refusal"] for part in parts if part["type"]=="refusal"]
        if refusals:
            terminal="explicit_refusal";stopped_text="\n".join(refusals);break
        calls=[item for item in output if item["type"]=="function_call"]
        if not calls:
            terminal="text_reply" if stopped_text else "no_tool_call";break
        try:action,cid=native_action66(calls)
        except (TypeError,ValueError,KeyError):
            events.append(dict(step=len(events),origin="model",action=None,observation=None,parse_error="malformed_native_tool_call"))
            terminal="format_failure";break
        obs=env.step(action)
        events.append(dict(step=len(events),origin="model",action=action,observation=obs,parse_error=None))
        # Each returned item stays in order, including encrypted reasoning and assistant message phase.
        messages.extend(output)
        messages.append({"type":"function_call_output","call_id":cid,"output":canonical_json(obs)})
        if obs.get("error") in ("action_must_be_object","unknown_tool","invalid_arguments"):
            terminal="protocol_failure";break
        if action["tool"]=="create_order" and obs["ok"]:
            terminal="order_created";break
    return dict(events=events,final_state=env.snapshot(),terminal=terminal,stop_text=stopped_text)


class OpenAI66:
    def __init__(self,key):
        import requests
        self.session=requests.Session()
        self.session.headers.update({"Authorization":"Bearer "+key,"Content-Type":"application/json"})

    def __call__(self,payload):
        try:
            response=self.session.post(ENDPOINT66,json=payload,timeout=(15,180),allow_redirects=False)
        except Exception:
            raise RuntimeError(" The network was interrupted. API  Whether the check is charged is unknown. Please rerun to check the checking request. ") from None
        if response.status_code in (400,401,403,404,422,429):
            try: code=str(response.json().get("error",{}).get("code") or "")
            except ValueError: code=""
            safe=code if re.fullmatch(r"[A-Za-z0-9_.-]{1,80}",code) else ""
            raise Rejected66(f"API  Request denied.  (HTTP {response.status_code} {safe})。 Check keys, model privileges, balances or limit stream follow-up. ")
        if response.status_code!=200:
            raise RuntimeError(f"API  Status  {response.status_code}， The cost is to be reconciled. ")
        return response.json()


class Rejected66(RuntimeError):
    """The server explicitly rejected the request before returning a completion."""


class Study66:
    def __init__(self,root,inputs,smoke,reference,assets,source_sha,provider,budget,retry_unknown=False):
        self.root,self.inputs,self.smoke,self.reference=Path(root),inputs,smoke,reference
        self.assets,self.provider,self.budget,self.retry_unknown=assets,provider,budget,retry_unknown
        if not isinstance(budget,(int,float)) or isinstance(budget,bool) or not math.isfinite(budget) or budget<=0:
            raise ValueError("USD_BUDGET  Must be positive. ")
        self.manifest=dict(scope=SCOPE66,endpoint=ENDPOINT66,model=MODEL66,parameters=PARAMS66,tool_schemas=functions66(),
              assets_sha256={k:hashlib.sha256(v).hexdigest() for k,v in assets.items()},source_sha256=source_sha,
              policy=NATIVE_POLICY,pricing_per_million=PRICES66,max_actions=6,bootstrap_seed=76102,bootstrap_draws=3000)
        self.sig=stable_hash(self.manifest)
        manifest=self.root/"manifest.json"
        if manifest.is_file():
            if stable_hash(json.loads(manifest.read_bytes()))!=self.sig: raise RuntimeError(" Tabs are tied to other protocols. Use new protocols.  RUN_LABEL。")
        elif self.root.is_dir() and any(self.root.iterdir()):raise RuntimeError(" Missing directory  manifest， Please use a new label. ")
        else: save66(manifest,self.manifest)
        self.states={}
        for stage in ("smoke","full"):
            for p in (self.root/stage).glob("*/*.json"):
                name=p.relative_to(self.root).as_posix()[:-5]
                doc=json.loads(p.read_bytes())
                if doc["signature"]!=stable_hash([self.sig,name]) or stable_hash(doc["state"])!=doc["digest"]:
                    raise RuntimeError(" Checkpoint validation failed: "+name)
                self.states[name]=doc["state"]

    def worlds(self,stage):
        if stage=="full":return self.inputs["worlds"],self.inputs["target_by_world"]
        if stage=="smoke":return self.smoke["worlds"],self.smoke["target_by_world"]
        raise ValueError("RUN_STAGE  Only for...  smoke  or  full")

    def persist(self,key,state):
        save66(self.root/(key+".json"),{"signature":stable_hash([self.sig,key]),"digest":stable_hash(state),"state":state})
        self.states[key]=state

    def usage(self,stage=None):
        calls=0;known=uncertain=0.0;counts=[0,0,0]
        for key,s in self.states.items():
            if stage and not key.startswith(stage+"/"):continue
            for call in s["calls"]:
                calls+=1
                u=call["receipt"]["usage"]
                known+=price66(u);counts[0]+=u["input_tokens"];counts[1]+=u["output_tokens"]
                counts[2]+=(u.get("input_tokens_details") or {}).get("cached_tokens",0)
            uncertain+=sum(x["reserve_usd"] for x in s["uncertain"])
            if s["pending"]:uncertain+=s["pending"]["reserve_usd"]
        return dict(calls=calls,estimated_usd=known,uncertain_reserve_usd=uncertain,
                    input_tokens=counts[0],output_tokens=counts[1],cached_input_tokens=counts[2])

    def receipt(self,key,state,step,payload):
        request_hash=stable_hash(payload)
        if step<len(state["calls"]):
            old=state["calls"][step]
            if old["request_hash"]!=request_hash:raise RuntimeError(" The recovery is not consistent with the history of the request: "+key)
            return old["receipt"]
        if state["pending"]:
            if state["pending"]["request_hash"]!=request_hash:raise RuntimeError(" The history of requests to be reconciled does not match ")
            if not self.retry_unknown:
                raise RuntimeError(" Last request completed unknown: "+key+"。 Check  API  If the amount is accepted for possible double-counting, it will  RETRY_UNCERTAIN=True  Keep going. ")
            state["uncertain"].append(state["pending"]);state["pending"]=None
            self.persist(key,state)
        # Operational reserve uses UTF-8 request bytes, framing allowance and the configured maximum output.
        reserve=((len(canonical_json(payload).encode())+2048)*PRICES66["input"]+
                 PARAMS66["max_output_tokens"]*PRICES66["output"])/1e6
        meter=self.usage()
        if meter["estimated_usd"]+meter["uncertain_reserve_usd"]+reserve>self.budget:
            raise RuntimeError(" The local budget gate was reached. Checkpoints are saved; verify usage and raise the limit to continue  USD_BUDGET。")
        state["pending"]=dict(request_hash=request_hash,reserve_usd=reserve)
        self.persist(key,state)
        try:
            raw=self.provider(payload)
        except Rejected66:
            state["pending"]=None;self.persist(key,state)
            raise
        got=choice66(raw)
        price66(got["usage"])
        state["calls"].append(dict(request_hash=request_hash,receipt=got))
        state["pending"]=None
        self.persist(key,state)
        return got

    def trajectory(self,stage,w,t,c,start):
        key=f"{stage}/{w['id']}/{t}-{c}-{start}"
        state=self.states.get(key)
        if state and state.get("result") is not None:return state["result"]
        if state is None:
            state=dict(world=w["id"],target=t,context=c,start=start,calls=[],pending=None,uncertain=[],result=None)
        result=play66(w,t,c,start,lambda i,p:self.receipt(key,state,i,p))
        state["result"]=result
        self.persist(key,state)
        return result

    def completed(self,stage):
        worlds,ts=self.worlds(stage)
        return sum(self.states.get(f"{stage}/{w['id']}/{ts[w['id']]}-{c}-{l}",{}).get("result") is not None
                   for w in worlds for c,l in cells66(w))

    def run(self,stage):
        worlds,ts=self.worlds(stage)
        if stage=="full" and self.completed("smoke")!=6*len(self.smoke["worlds"]):
            raise RuntimeError(" Please complete your independence.  smoke， Look at the actual costs.  RUN_STAGE  Replace with  full。")
        if stage=="full" and not any(e["origin"]=="model" and e["action"] for k,s in self.states.items()
                    if k.startswith("smoke/") and s.get("result") for e in s["result"]["events"]):
            raise RuntimeError("smoke  No parsable native tool call was received. Check the model and interface first. ")
        for i,w in enumerate(worlds,1):
            t=ts[w["id"]]
            for c,l in cells66(w): self.trajectory(stage,w,t,c,l)
            meter=self.usage()
            print(f"[{stage} {i}/{len(worlds)}] {w['id']} | 6/6 |  Cumulative recognition  API  Call  {meter['calls']}， Estimate  US${meter['estimated_usd']:.3f}",flush=True)
'''


In [ ]:
V66_NATIVE66_REPORTS = r'''"""Blinded text review, paired estimates, offline replay and verified exports."""
from collections import Counter
import shutil
import numpy as np

METRICS66=("final_y","final_x","fallback","other_order","no_order","strict_success","first_free_check_x",
           "first_free_check_y","first_free_reserve_y","later_checked_x","reserved_y","attempted_order_y",
           "format_failure","tool_error","explicit_refusal","clarification","text_unreviewed","length_stop")


def audit66(inputs,smoke,ref):
    tmap=inputs["target_by_world"];worlds=inputs["worlds"]
    assert [w["id"] for w in worlds]==[f"confirm-{i:05d}" for i in range(97,193)]
    assert len(set(tmap))==96 and set(tmap)=={w["id"] for w in worlds}
    assert len(smoke["worlds"])==2 and not set(tmap).intersection(smoke["target_by_world"])
    for w in worlds+smoke["worlds"]:
        t=(tmap if w["id"] in tmap else smoke["target_by_world"])[w["id"]]
        x,y=w["items"][t],w["items"][1-t]
        for start in STARTS66:
            runs=[messages66(w,t,c,start) for c in CONTEXTS66]
            lex,rel=runs[0][1],runs[1][1]
            a,b=lex[1]["content"],rel[1]["content"]
            if (a.count('"attribute":"preview_item"')!=1 or a.replace('"attribute":"preview_item"','"attribute":"preferred_item"')!=b
                    or a.count(x)!=1 or a.count(y)!=1 or lex[0]!=rel[0] or lex[2:]!=rel[2:]):
                raise ValueError("Exposures, label or visible history differ unexpectedly")
            for env,msg,events in runs:
                assert len(events)==int(start!="natural") and not env.state["orders"]
                if events:
                    call,result=msg[-2:]
                    action,cid=native_action66([call])
                    assert action==events[0]["action"] and action["item"]==(x if start=="gold" else y)
                    assert result=={"type":"function_call_output","call_id":cid,"output":canonical_json(events[0]["observation"])}
        if w["id"] in tmap:
            for c in CONTEXTS66:
                for l in ("gold","foil"):assert type(ref["final_y"][f"{w['id']}/{t}-{c}-{l}"]) is bool
    assert len(ref["final_y"])==384


def review66(path):
    if not path.is_file():return {}
    with path.open(encoding="utf-8-sig",newline="") as f:
        rows=list(csv.DictReader(f))
    result={}
    for r in rows:
        rid=r["review_id"].strip(); label=r["label"].strip()
        if not label:continue
        if label not in ("clarification","decline","other") or rid in result:raise ValueError(" Malformed manual tag format ")
        result[rid]=label
    return result


def row66(w,t,c,start,state,review):
    result=state["result"];events=result["events"];orders=result["final_state"]["orders"]
    obs_orders=[e["observation"]["order"] for e in events if (e["action"] or {}).get("tool")=="create_order"
                and (e["observation"] or {}).get("ok")]
    assert orders==obs_orders and len(orders)<=1
    sku=orders[0]["item"] if orders else None
    x,y=w["items"][t],w["items"][1-t]
    case=case_for(w,t); m=gx_score_actor(case,events,orders,case["initial_env"])
    free=[e for e in events if e["origin"]=="model"]
    def use(e,tool,item):
        return (e.get("action") or {}).get("tool")==tool and e["action"].get("item")==item and (e.get("observation") or {}).get("ok") is True
    first=free[0] if free else {}
    terminal=result["terminal"];rid=stable_hash(["v66-text",w["id"],t,c,start])[:20]
    text=result.get("stop_text") or ""
    label=review.get(rid,"") if terminal=="text_reply" else ""
    row=dict(world=w["id"],target=t,template=w["template"],reverse=w["reverse"],context=c,start=start,
             final_x=sku==x,final_y=sku==y,fallback=sku==w["fallback"],other_order=sku not in (None,x,y,w["fallback"]),
             no_order=not orders,strict_success=m["strict_task_success"],terminal=terminal,
             first_free_check_x=bool(use(first,"check_stock",x)),first_free_check_y=bool(use(first,"check_stock",y)),
             first_free_reserve_y=bool(use(first,"reserve_item",y)),later_checked_x=any(use(e,"check_stock",x) for e in free),
             reserved_y=m["accepted_conflict_reservation_item"],attempted_order_y=m["any_order_conflict_value"],
             format_failure=m["format_failure"],tool_error=m["tool_error"],explicit_refusal=terminal=="explicit_refusal" or label=="decline",
             clarification=label=="clarification",text_unreviewed=terminal=="text_reply" and not label,
             length_stop=terminal=="length",api_calls=len(state["calls"]),review_id=rid if terminal=="text_reply" else "")
    assert sum(row[k] for k in ("final_x","final_y","fallback","other_order","no_order"))==1
    # Human coding sees response text only. Confirmed clarifications stay distinct from implicit text replies.
    blind=None
    if terminal=="text_reply":
        text=re.sub(r"\b(?:req|sku|lot|acct|evt)-\d{3}-\d{3}\b","[ID]",text)
        if text.lstrip().startswith(("=","+","-","@")):text="'"+text
        blind=dict(review_id=rid,assistant_text=text,label=label)
    return row,blind


def effect66(name,values):
    a=np.array(values,dtype=int)
    sample=np.random.default_rng(76102).integers(len(a),size=(3000,len(a)))
    lo,hi=map(float,np.quantile(100*a[sample].mean(axis=1),[.025,.975]))
    return dict(contrast=name,n_worlds=len(a),estimate_pp=float(100*a.mean()),ci95_low_pp=lo,ci95_high_pp=hi,
                positive_worlds=int((a>0).sum()),negative_worlds=int((a<0).sum()),tied_worlds=int((a==0).sum()),
                degenerate_bootstrap=bool(np.ptp(a)==0))


def analyze66(rows,worlds,targets,reference,formal):
    groups=[]
    for c in CONTEXTS66:
        for start in STARTS66:
            g=[r for r in rows if (r["context"],r["start"])==(c,start)]
            groups.append(dict(context=c,start=start,n=len(g),**{m:sum(r[m] for r in g) for m in METRICS66}))
    if not formal:return groups,[],[]
    grid={(r["world"],r["context"],r["start"]):r for r in rows}
    assert len(rows)==len(grid)==96*6
    details=[]
    for w in worlds:
        wid=w["id"];t=targets[wid]
        def y(c,l):return int(grid[wid,c,l]["final_y"])
        def old(c,l):return int(reference["final_y"][f"{wid}/{t}-{c}-{l}"])
        dy={c:y(c,"foil")-y(c,"gold") for c in CONTEXTS66}
        g=dy["C1"]-dy["Clex"]
        old_g=(old("C1","foil")-old("C1","gold"))-(old("Clex","foil")-old("Clex","gold"))
        details.append(dict(world=wid,target=t,primary=g,foil_label=y("C1","foil")-y("Clex","foil"),
                   gold_label=y("C1","gold")-y("Clex","gold"),natural_label=y("C1","natural")-y("Clex","natural"),
                   C1_start=dy["C1"],Clex_start=dy["Clex"],baseline_8B=old_g,strong_minus_8B=g-old_g))
    effects=[effect66(k,[r[k] for r in details]) for k in details[0] if k not in ("world","target")]
    return groups,effects,details


def csv66(rows,fields=()):
    fields=list(fields) or list(dict.fromkeys(k for row in rows for k in row))
    f=io.StringIO(newline="")
    w=csv.DictWriter(f,fieldnames=fields);w.writeheader();w.writerows(rows)
    return f.getvalue().encode("utf-8-sig")


def export66(study,stage):
    worlds,targets=study.worlds(stage);ready=0;rows=[];blinded=[]
    annotations=review66(study.root.parent/"text-decisions.csv")
    if json.loads((study.root/"manifest.json").read_bytes())!=study.manifest:raise ValueError("Run identity changed")
    with tempfile.TemporaryDirectory() as tmp:
        local=Path(tmp)/"snapshot.zip"
        expected={"manifest.json",*study.assets}
        with zipfile.ZipFile(local,"w",zipfile.ZIP_DEFLATED) as z:
            z.write(study.root/"manifest.json","manifest.json")
            for k,raw in study.assets.items():
                assert hashlib.sha256(raw).hexdigest()==study.manifest["assets_sha256"][k]
                z.writestr(k,raw)
            # Include smoke trace provenance in the final bundle, without including it in the formal denominator.
            for bucket in (("smoke","full") if stage=="full" else ("smoke",)):
                ws,ts=study.worlds(bucket)
                for w in ws:
                    t=ts[w["id"]]
                    for c,l in cells66(w):
                        key=f"{bucket}/{w['id']}/{t}-{c}-{l}"
                        path=study.root/(key+".json")
                        if not path.is_file():continue
                        raw=path.read_bytes();obj=json.loads(raw);st=obj["state"]
                        if obj["signature"]!=stable_hash([study.sig,key]) or obj["digest"]!=stable_hash(st):raise ValueError("Corrupt receipt: "+key)
                        if (st["world"],st["target"],st["context"],st["start"])!=(w["id"],t,c,l):raise ValueError("Wrong branch: "+key)
                        z.writestr(key+".json",raw);expected.add(key+".json")
                        if st["result"] is None:continue
                        used=0
                        def replay(i,p):
                            nonlocal used
                            rec=st["calls"][i];assert stable_hash(p)==rec["request_hash"];used+=1
                            assert rec["receipt"]["model"]==MODEL66
                            return rec["receipt"]
                        assert play66(w,t,c,l,replay)==st["result"] and used==len(st["calls"]) and not st["pending"]
                        if bucket==stage:
                            row,bl=row66(w,t,c,l,st,annotations);rows.append(row);ready+=1
                            if bl:blinded.append(bl)
            balanced=ready==len(worlds)*6;formal=stage=="full" and balanced
            groups,effects,details=analyze66(rows,worlds,targets,study.reference,formal)
            def zero_bound(natural):
                group=[r for r in rows if (r["start"]=="natural")==natural]
                if formal and not any(r["final_y"] for r in group):
                    return 100*(1-.05**(1/len(worlds)))
                return None
            report=dict(scope=SCOPE66,stage=stage,status="COMPLETE" if balanced else "PARTIAL",
                        endpoint=ENDPOINT66,model=MODEL66,reasoning_effort=PARAMS66["reasoning"]["effort"],
                        world_count=len({r["world"] for r in rows}),trajectories=ready,expected=len(worlds)*6,
                        manifest_hash=study.sig,primary=effects[0] if formal else None,meter_all=study.usage(),
                        meter_stage=study.usage(stage),conditions=groups,effects=effects,
                        zero_controlled_any_y_upper95_world_pct=zero_bound(False),
                        zero_natural_any_y_upper95_world_pct=zero_bound(True),
                        label="C1=preferred_item; Clex=preview_item. Bootstrap resamples independent worlds. No-order stays in denominator.")
            tables={"actions":csv66(rows),"conditions":csv66(groups),"paired-effects":csv66(effects),
                    "world-contrasts":csv66(details),"text-review":csv66(sorted(blinded,key=lambda x:x["review_id"]),
                                                        ("review_id","assistant_text","label"))}
            for name,raw in tables.items():
                name="reports/"+name+".csv";z.writestr(name,raw);expected.add(name)
            z.writestr("reports/report.json",canonical_json(report));expected.add("reports/report.json")
        with zipfile.ZipFile(local) as z:
            assert z.testzip() is None and set(z.namelist())==expected and len(z.namelist())==len(expected)
            assert json.loads(z.read("reports/report.json"))==report
        digest=hashlib.sha256(local.read_bytes()).hexdigest()
        dest=study.root.parent/f"{SCOPE66}_{stage}_{report['status']}_results.zip"
        fd,temp=tempfile.mkstemp(dir=dest.parent,prefix=".v66-",suffix=".zip.pending");os.close(fd)
        try:
            shutil.copyfile(local,temp)
            if hashlib.sha256(Path(temp).read_bytes()).hexdigest()!=digest:raise IOError("Drive  Temporary  ZIP  Validation Failed ")
            os.replace(temp,dest)
        finally:Path(temp).unlink(missing_ok=True)
        if hashlib.sha256(dest.read_bytes()).hexdigest()!=digest:raise IOError("Drive ZIP  Replaying does not match ")
        save66(study.root/"reports"/(stage+"-report.json"),report)
        atomic66(study.root.parent/(stage+"-text-review.csv"),tables["text-review"])
    print(stage,report["status"],f"{ready}/{len(worlds)*6}  Tracks ",dest,"SHA-256",digest,flush=True)
    return report,dest,digest


def finish66(study,result,mount,enabled,is_mounted,flush,release):
    if not enabled or not result:return False
    report,path,digest=result
    if (report["stage"],report["status"])!=("full","COMPLETE"):return False
    mount=Path(mount).resolve(strict=True);path=Path(path).resolve(strict=True)
    if (report["trajectories"]!=576 or report["primary"]["n_worlds"]!=96 or report["manifest_hash"]!=study.sig
            or not is_mounted(str(mount)) or not path.is_relative_to((mount/"MyDrive").resolve())
            or path.parent!=study.root.parent.resolve() or path.name!=SCOPE66+"_full_COMPLETE_results.zip"
            or hashlib.sha256(path.read_bytes()).hexdigest()!=digest):
        raise IOError(" confirmation  ZIP  or  Drive  Mount check failed, keep  runtime")
    with zipfile.ZipFile(path) as z:
        if z.testzip() is not None or json.loads(z.read("reports/report.json"))!=report:raise IOError("ZIP  Readback failed ")
    flush(timeout_ms=600_000);release()
    return True
'''


In [ ]:
#@title  Built-in formal input, history check and independence  smoke
V66_ASSETS={}
V66_BLOB_0 = zlib.decompress(base64.b64decode((
'eNq1XU2THLmN/S99VkbwAyRAXfe6u977hsOhGbcditGMbKnlXYfD/33xmEkWO4F07EwrTx6Vq7tZSBB4eHhA/ePp5cOXPz+//OGHv//hfz5/+fTHp/f/ePrx'
'8y9/+vjl5y2E0PjpfXj36hV5eh9fv9JOr8QQTj8VQzTvSeY92byHzCvF/FQ1r7D5KTHvMWeO5swxmleS+Slz5mjOHM2ZYzXvYfMec+bYzq+kcP49ydg5mTMn'
'c+ZE5jcX8x5j52TOnMT8lDlzNmfOxs7Z+EbO5hVj52zOnM2ZszlzNnbOxjfInJnMmcmcmYydydiZzJnJ+AYZfyZjZzJnLsafizlzMWcuxs7F2LmYMxdj52LO'
'XIydi/GNauxczZmr8edqzlyNnau5g9WcuRrfqMbO1diZzZnZnJmNndmcmY2d2ZyZjW+wsTObM7M5s5gzizmzmDOL8WcxZxbjG2LOLMbOYnxDzJmb8edmYl3D'
'mf/57qlnsq9P7/9bc9nHl78/vX/6949ff/j8y5O++9vXl88/P3/R1z78+OPLVkrZmsatd0/Pf3v+5QU/pP/1stVCW9Jo+q7/q9Wykeae3797+tOHT59++PDj'
'T/oLvv70bZMYN9bI8+7po+bOU+rUF3/54/P/Pr1vVf/z5fnn/uvxYyyykYbmd/1ftcrW9KHor//0eT+D/q+eizbcgP5qP2nVe65v+vL812/PX/c36n/ry2VL'
'euH7/7MVDptouO5v/Nvzl6/PT+/11F+f9Y/95dPHl8cx9Qf0UH/59OHluRtumuu/vnz487dnx1wcw5ZrsebitCUpw1xEW+PqmKtK0rNFz1yymItP5iqUtix8'
'mKsU/WONjLla3Kom9t1ctdUtK8hwzMVB9JnVw1yS9bdR+LXmiou5/uPbzz98+OiZS/LWojVXCbyRRtbDXCVvmTzvaok3qa53tcVccvYutURQOHV4V9MPGNPZ'
'XJRy9+ndXBLbRrV45mqBtpzlMFcNuwes5nr58u1XONfvvn746YNjrBZpI8nGWBz08kk7jFVT079P3lXUByn9FpyMBTz6MFY7+5Y0/VA0fEstTopszr5VWN+Y'
'h29Fdcfq+xYuH9NhLNaLzdJ+pbHi/+smZr3z1VpL9GkXnoErVLUYu66VN8WUnrXiw1pqupO5qvCGBHv4lugpFI6cfauGLfao2iOXvikUcc2lvy2XMswVBSZ7'
'S+S6vIpU0wYIZCJXiBpexlWspJdCxLFXUXtJ/0jGXmm1VzzbKzc1wIz0+gmlmtBFTX95bdNeSfNPdu2Voz7TYS88aySfN4Suy8SIZ0FeYtRbVuu4jaWxJkfP'
'XhTrFjJ79sqrvdLJXi0lzYVy2Iv1OqY9l632qupfzDN2ZQQON3aRxpTSHbpnxkoj0P7G2HUd6TXdjAT46jriupAc5mLFEeyaS4Jex5I9c9FqrmxCvbpXG6Ge'
'qwZHbsZcSb2mzuil6U+rHNe9NLMCLw5zafAo4Q3R69JcVW8jFWsuorCFVsdt1ANkRQtO9Gr7k3TMVVZz0dm7FEioT41gn7NmF4O7RC8ZE09zIeEF17tYH3rO'
'I3rpG+sOS35r9LpKjUU/bmBxcIRG1jwvoz594HYHdulBcw9yxlx1NVc5mUuyhuc4ghewLiqOk7mK3tjUsWw3njp8vTBX06wwzVUVJIZEbwle//nh45fPP3j+'
'VTQkZK4OrC/AsCM75qqxxjOY6GcK7GZHXg12xvWk6T9JPQxGGkMP13l1HYERegrdDdYQPXygqjiOecH1Z//6ddHrEkwU1Dce9FJYX3lEr6bPn2Lz/EuTAjgR'
'x1yymuuM6ykDB7dhLlZLBONforVSa+M6siIx8s1FCQ93JEdihLnyhuj1b3Avx1qiYYnYBi9NzBoOhrUK4e641lKrogZ3rNVWa51hfUXQ6xCmu2jPJcnGen05'
'jdRIzIqxXGtVBZBtOhciv1rsLcHrEko0QfSy3lUET+hxGfXqFK9oZPUSH9jHsNrrjOwFUbMN72pai+Vsq0a9fjmNy0iVkVvcy6iZlssog5o+Q8rhnujVFDsl'
'B6sKXPrA/PoujQfI6h6YiOp6HpiIK7aPwYR7TSd1RC/A0BSMwQoQhwxsr2W+/jk33KPW35NWx6pMepPDG6LXNVQVgFCy/kXgCQaYkAZo4WIvDdOhJwtjrhXa'
'xzO0F00muQ7sRQTAa8zVmZ44gz1ypU/iAIwEHiSOBPWvEO6oHItCzRqsd5WoB0iDwyEQYMGrswG+o5sb44rs4xnZF0atNytH9XGJJtiDykLUPII9ig1iH9nn'
'LdYR7BFi8iiZvnPliMAUvEobtUWd4V7vZss+VuWLyjGu0D5mJ9xLnNBe7UUh2fClEKMNjrCVuAm5lbZEoLRxGzWDbjG+qXK89C8SVHDedQwHpur+FXfU6PtX'
'9a/jiu3jGdsXRXOUJvYKKLya5VSzHmT4V9M7N6P4yb/05zkMIkfLIK0d6h3YC67bWrLYS4PxnvX26KUfp7nuVdQDOj1mzLVi+3jG9viAOcXpXmVDx+pMTJAm'
'PZ7RSz0S9JAX7BnYhyf2SlspckflKIQiwuFx9MHyIzdq+MzJC/ZVP1El9zauyD6ekX1R3BnyzI1lJyBM9NLEzZOCThr6/eglWv/m9qCgmwKhdkv0Kgp7pNlK'
'iOBQM3r1uojjhb2ieJV2XKF9NNC+RoSj4V5aj9VseRz1OulXvSMLTUyJXPBV9Hm2NogJ0gKd+E3R66rSZs0/6mI2eDEwxgheFbm6uuYCOuLkmWvF9vGM7UVr'
'By6TstffX3fi4xX00ggXW57BSzMo+di+NE3gYSbHppAy3RG8JOnHzdZcDe2EOPtBEhWI+eaKx606myut0D6eoT1o9tLCgPZqPKnRdjhAkYdxG2tTK/vQXkMw'
'1+FdrJlcy6E7ghd4HAk2NVIATZlGrMe/fBJaf57ZC15pBfbpDOybgEibtFdvXxjoxWkwvv1fKMaj2+MoTQ0Up7mCOmUNbwle/6ISSvqcg0N86ethtmeZci8w'
'/VI7uV2OtEL7FE30qgdvvNOqioXZdtDQ3ZlYQrSCbtlvCqllax3Rvulpa8tviV5XzAQrJE3NoVX1mgYZ/tX0+R2l2NlcmkRb8ArHtEL7lByoGiZUJU057PTQ'
'1KPKbGdzQrfOjV4KoxXMPwrHiNrohoYj6FvqmOXc4hi0W7edukJklySM7YCIxlorsE9nYC/EPXsc1sLTsIVQ00fWHs6lNUihctVCKzKhhGZUDm9BXlfWkowO'
'hxPq4TM8gRfJBSnBqMbZw6lphfXJwHoNxylNjlA9jXK0DSGt/+JsOOojm9TfyVroaUzgpQFOA12+J3aBFBWHJmRUFm3kRmRA8ZEqeqPV66ClFdinM7AHlisT'
'qeLqVzYcdNGoXR65sdQrYC8RmGPU2WgksNxEe7HehNqcQpvQ45pQVSuNJl5LG13UPcwag63QPhlon3ei6wATDa0pcx+1TNz21niPXlUvXQy+AqBoPTYL7RK6'
'NOcGuYSiOkq2EOKYj17PTuqAQfGg106ru9ZagX1iJzWGOhu06hDF1o21RP2zoyfUtGZrgX0swb3hfriXpvUU5QbOvklxy6BStebgwRFyhwteZmxVP0L0OOi0'
'4vpkpDghHi3CHb9pMRbtZSzoSsapLmG9Br61GA3LB1AtWxj15Xfm7JF1W3ZIHBS+U+glai+uPuklW8xetM8rsE+Ws0flM6I9uhIlRk/plWOd3hWvghf67a0O'
'7yoayrilW+QSoCVK9cpszZpx3EbWuiwVLzsSEDp5tzGvyD4bZK+m4DT9Sz9hscGeq5qxjZ5Qg7KjXTS0C67usBchyJU76kaocYqnI9RKLs66sUvzgudecIwU'
'XPdacX2O5joS1HOzJRRGFH91HVl6n/PdhBYSfPUSKnAZpFcLDeH2BugFWUdzEiPUV4XSDF48WKSzc2l6Dy70yiusz8kpG9tyGSPqKXMZ0TagOilChRviVtma'
'bHoqGkgCXYR7tF6IoKmKkxp52xv4O7DXg2e3IYR8Fd3LuAL7fAb2mo67Gm60szN0g7Z/pndsNoQA1q7KIK0TWx72agWq3nZH1ShozDjIXtBKznnBXRK9OqgB'
'n2WPlcgrss9nZC9aP6cHRagZo5BtCKF/9lD1xogL6cYuhcsUhroEKVVE3hC7rqwFNeOuKD3z9Q20xFR61SFcO19Gjari4q68wvps+PqoTjnbZ8gXwUrGmTTF'
'0YD1LCAjmy/qjZAJDyQBmV28hfLSZ7eFFp2yUYHpvIuAC0fYMEqcBn7HM9cK6rNV2CuwiytfX5vh64uEo6YdknEhVxjXkkbCyeBw143do7BHOC3EbuwKYRZB'
'fWLAhfX4EMmN9Susz2x0lwqVaFKEDf0AmxkFRNgAqnvH2wUS0sVdo2pEgQXXvaVqxJ9KHBwpoUaXPJA9OlISPeSlCAep2zPYiuyzGOTVOpY/oEQWNNCtlDDo'
'bxraJWhiJecr5MWpTqSqiepN0etaNy54zk5uxIgFTd14Tr2B4LASAcyrF+xpBfa5GVYCcXyEL5bk+Rfw+i666OaCuLz4rITixPpg7BXRFuY3hK9LXN/bhY52'
'iYH3p/ISao0cXK2XRvtWvGhPK66nYDiJ2Gu7w1yE0ZZsoVc5Wrg765XV6V0CGrB6b6nvNbfCFKabGPtIwCpO979tcTL2pAmnuumxkzG+wVZkT0Zn3/iQz+zQ'
'ovUu8ZnEwaxHGvG+JDSp/PiVkAoe3X9+iJy+b7+R6lCVnbWE3OdxBlQtQ65xLoQwK8AeL0ErtCcrsw9QxE1lbwZ+8ZS9LG3tCPkNDrSzKz2uY8EBbuC8agSb'
'5lmL+v0b03q0RR97KSqsLvaiFdjTGdhDY8Fdrj5snsTGemhqppIQcx+tZf821s5gHgxhn23Ltyi9eBCiZ+w1JHwdS3Rk4Q43Sjw6gsZcK7AnA+wZaClNQnUK'
'YV+PVJXOhRx3UeOd5OZ3z7jPMB11EOIDp3uCF2mcZbZglQBgKEwswZ2F89rZosnRvY0rtidD2Wfg7zKlS60res7+BeI7hyU5XgwJtThQ7c7iaJJ425DQNbgv'
'mklCdaZeyiGD31lC7mIXnyUUV7tEK7gnI7PHhw8y1RKp9/nPYAJjng8H00c7McJZCZ16vBqsqmaF8Bap179oCdWolnDAfUBzQaZ/YRpM3EobIievJUQruCc2'
'4Is0pc3+rMZHtsJetdXBmuwca+2iWo+FhkCv8ew45qFM+97xq3bhtwVfKH5LfXT/1dmC20HTIpyz16ClFdqTEeOoC+zjeDsLlvA3jLm0CMyPWihDHubPoGkl'
'Guf8LFTVHPiOKSFkN0pO9x/i2BnukatT8vtn6p3Jy45lhfbUHFJValiGXqpVlyBF10mqQskOssQzlwJFmcwEen8U8j3RC8M6ZHscDX2VNKez9eCYUfH7jYeG'
'92yvFdsXO0KraH4OCWHiuNhoj+iVZilUIQevbilUCLDjIYSOjxLzO8vsgSYcc2H0McS0MBOxeMwExtOoelqcsiL7ckb20HJKeujGQ5+nNeISzbw0wFdLClya'
'jya0SuIoE6q2HpC/f8ORMkGu5dISMoeEQKHk7NZB+iB3WbCx1grsi5HiIIqX6Vz616KlJdBnbXNICDzsBU2IeYVWBqkqatZY41ti1yWrqunZFUuoGePU4pQC'
'vODRhOiGx+ohibIi+2K0OIIUM+pszIgky6rWzsSPu4hBIPG72ZTl2Bqyk4atq/XuENmj9RS91KgPnPJUeimqLj5S7QI3j1YtK7QvZoAWsX7GrqZFDtmJ0Brp'
'kK731AgJRPQHHKETn/1szH7UFm5BXoSTitdw5GNMfZ9I039d5MZ0CA6NvVZkXwxr3+LhOru3gXGw9oK2PA6kqqVFH8J2W0JyTLTtkV/dst5CewnayqXZ6ygB'
'VOqsHHMf4HNoiZouWPuyAvtiWPtEEDGOWE/+tgQNmtRmS0jRDXH0CyHuI1mHuSBUDeEWuQQKx1ySQ0zIFGTuAndyO4699VBde63AvrCxV+s9xsO9knRpzjk3'
'QjzYRm4s6OP5rD2EvWmS0H1Au95TOGLSMElz5CU8JWC7go5doWoH39m9jiuyL2LES1AiyZRL7Fz2GXod+tgxlYBH6K9L0LJNmixDL63eE776IqGcHGKC+8j/'
'Ae0xDFYuw1fwiIm6QvtyhvYNQ0g85SUaoEtwmo5ol88hIQzoJL9F2zf0PPQSFVN8d4i9cCuYHL2EHk3ynNgLQxlprIX5Ww971RXYVzM/C4XDFHthKqHYcXbC'
'tieeyzj6eKg/UpW5dxlHzzH0JvsNdSOH3KX+JnhBrdSmc/UJr+QCezm2hhhzrcC+GsoeuLPNHkeBjthqvbIsHW1IEpKIXzdmtGlXc4Xb5rO1lk1O4QghRKkL'
'mEhSXGUvHdNfxmArtq/JRHvcxrD0aIvYGbSK4agw57Pz1cBx335RaMrss8bDdkv0AkUUPJpQP0Hih9qrjSkeUwsRBiA9e63gvprlOL0j9hh70X+l4A1VFZq1'
'kP6po3Fk9UtySB/77Yxx0CjfOXqVBCGMbXIQVg22AVX7eCi7m70KdZ2WY60V2lcD7TFGL2E2OTBClr1VQstAaNS0Qj4tEdE4nbQEZL5vY70u1Us5zVGg18he'
'g32dq4RQkvhQFSI6d6aqrsi+2vnZDK54Vo7UdwYZkhDTqpOWoEmNecG+xsl6QWBC7Z7oBazavHUc+AhhYi+NNZnlookm7rKXumL7Wh2FCU1FDoSCic1EKAU5'
'Zn/2XLn3EVxsj10VczdOgrS33SD3wmP2ptkxj1+XjZeYkvBXL+1rRxxrrci+Gpk9tJehzUqIx3oWI+ydlD04FLpYvYRKaG5LAI6jlO4oHKmr+BzoBR1Enuo4'
'qM+CO7BXahc2OeZagX01OvsivQlwmCtwZ6QNpxq32MJSOF4UQpjsyx287HuF5EH4fO+FqpWxZcsphBpYzBm9areet/QyHFt8zvbiFdhXK8cZ8wz9ajIN3uMM'
'JWpLi1T1giaENpQoLC2OInwL7wXVFTktbSwJLZP3AqPJbqGNZlFw1V68Qns2OvuKYa7HQKgW2tkqAGTfOjAW9jKupM/jYJfvYwotd6XaDZx9gTIqO8iLgE0f'
'A6HUh1Q8BQC6Vd70P6/Qno3OHsoBGeaCdrk5LQ6QPXN3CTENLtHb9UJ50F4dokm6o+HYatMw6dBemKePeTYcsc8quViiHa0QY64V2HMyLKH+HA2WsGiYUjxg'
'C0c+8N+uJQx9aZGr7Q3H6oVdvIQu1S2FI2QxXrCHzCrNHaGYcDy2lBqkWo6pTWOuFddzdqSXqT7mEloX53ibXtLc4ljilRyHQ9riXF0CFJSIb9ktgcEbTxoX'
'MHU2gH3nM90xDmzequ68Ma/AnsmZ2JOHdyFjFLuKA6twylxBi13MfvdfC1+UKFMtwQoT6ZayEduv2elxYLtFTnPrZSel4gWpGn17rciei9mpip3wvCwSOpZX'
'vO5n12MzwV5EUp9H8gqhNLaQDaVck3KH0h5c8FDwvt5T1bCWZO7iqD3ye7E+H6ukjLlWXM9GjKOWyFzmRmg8NLItIaTGPKeEwmNx8XneOC7bxlFWpbftVL3u'
'cWBHCdlgD21xeNgL6jMXqnKMB8to7LUiezbL7Dt4mAvHFTpFy0EXEF0zeimWwX4J114SjhWt+xpHRbg3ib26gtcZCUWgTbOFhm3zUtmd5GBgIs9eK7Rns/aS'
'47IkFGqcQ5n+WrzE264j2FMlllT7M3toIc8F7dj3GN+0Guda2Iu672K5RJzaJQybu7mxi7RcFkdWYM8G2IP2yOM2FkXwxe7+36uxKVXFI7uY2UNunateGuaU'
'Y7ujbiyog5zghbXrHOZ8dprhwHDQaL54QFVWXC/BAaoy141XxjY9Z509oMRcsqd+3PxvlhC9p4/LSH21Q7pnrVcXxQbHvYZSeacM8wUFjeC3U6/GXiuwFyuz'
'Hwuf9uAVIE0ydRAALD8WaNexi80qCSEpnw0hwRAY37RcotKhPjoPOSqYl8e6BBnP1uFxmttBkxXai+HssVIgTDBBcUw0v7qPALRzfQlGvdife8G6BXkUjliY'
'Vu/ZSogBCmeBNniJx6YqrHgt5FZC+AoYd/GSrNBe7G4cKFbmlGPevy7m7F962YNMyh7XrPgNbb24MgUAqLK43oO9sFrAKRyRX+pjMCFyXxjryXEwmexexxXa'
'i91nPzr2u5SweLmx9FnRMrcSZojo/J2XsS/zGFiCuh74lm/j6CKE5AjHMRfGc6wq94XX3joO1o/kVY6yQnsxpD2wijxuo5rC4SWw02R+XQJEFTX5yl7odh7r'
'JfRMXG+ZEsIXfUTne4S6PK9O9RL267OrVNWjJXfNnqzQXgxlD96R4gQTAfffQnvuo49jCi314swd5NBc3sKUS9Q+EPj9oRdo3uaNvej5hebMXsIX5bgzVdih'
'VTxgLyuwFzaxi3rcHmKcPFYSnxto5bEZR7FtvPpqnHBE2tF8DHzLNwlhCW52utlYvpwyTSjRxrDv2bly6WN0jrlWXC92Nc6+OnxsS9gn7hykGuZGVfwIpXr1'
'HWgLZY9dGCHfolTFA3R27PXxS36IS9JFexbt3ewKe9sK7MVsxsEi1ziRas/DzgyaHKNl+4yQ1hrii0vwBSwyJ/bQnmW6Zx00VnV6LA6+imUi+y4Od7ez49vR'
'oislbCuyb0aMg7GQucy+aWzMzvpsqn0qaEi90hi8sYVQOL57YefvaWxL/t6MPQ7asjPEge0ycz67D0C1cKEb9/uNbQX2zajssYxjbrzsk39WqKp3bwsPJWGu'
'fbXDxe4Sbquwl3/DfPbv//l/oxqEQQ=='
)))
assert hashlib.sha256(V66_BLOB_0).hexdigest()=='5d9c74662f1a0ecf5de144113cc06607c8cbe4d51ba3faf2fb62de8f98ed88b9'
V66_ASSETS['inputs/formal.json']=V66_BLOB_0
V66_BLOB_1 = zlib.decompress(base64.b64decode((
'eNqVkd1uwjAMRt/F17UUx2ma9CH2AhNCpWRTRFlZG9gQ6rvPDWM/wMXWq9g9dj6dnCA1w3NIy9Vx+dYP3RrqE+xi1ydU8hHUqvhRa6hpKiCTI9SPJ2hjOkIN'
'D00c+lWEAtr9mPptGKTZtG1CtoxOW/kTDuElzVNySqiNRWdV7ktFHlXlYVHAU9N1q6bdyIJxs0dWJPNauCjhfmWT1ss6vOeMMYVt3j3PaE+oShYgb5AExpl5'
't8xmaN6hlRdwDpYrsqg5Q0N43YfxDMp5htC7EopzpRipdGfwEIYxQC2RxyCX7bqYLhkFl0i7rkkCqKn4gyptHZIpb1WxwUqbL1VGQug7qrQ16MjdqtLfquha'
'VSXPQ+VFlauwdNW1KrbyVswXVUYjE99V5Tx69p+q2FTyxPw/VTQtpg9LPL+C'
)))
assert hashlib.sha256(V66_BLOB_1).hexdigest()=='3b1a91945a5a3e21f6964d9b32cf76a6fa4ba51c3369c8ca254ef2a474ba644b'
V66_ASSETS['inputs/smoke.json']=V66_BLOB_1
V66_BLOB_2 = zlib.decompress(base64.b64decode((
'eNqFmk1v2zgQQP+LznEjcj6ZY/e6/yFQHHkbwKkLpw1aFP3v2wWSxgtE8y6+kCIGQ+r5zVA/p+W8//TwvN4+fVq6+XQz7e3OVlnmdZ73Le/msSx3+/sxt6Uv'
'4fd5N6TvQ2XO0MM+93M7pPXhKWtTtbvpajo8fF6Otz+mm5/T/vT58HB+3M3zPOJ63v3VdofTw3G6+Xr+tl69N/zP6Xg/3RyW49N748f1+8sC1YzNNfK6VSG8'
'DNePv4WwsUAZwagjGBDBoAgGRNDmudqGP8P149sRXEzYXKJVOfgzXD9eHISLGZtr9DoJHZLQ4SxezNhcQ+osCGRBMAuCWdA6BIUQtD6MFxM2l7B6Hwz2wXAf'
'DPfB6xAcQnB6HxwjiHobArYhaBsCtyEvc7A9Xj9f7kNiFv5Hxu3x+vkyBmRjq9nYgI1tpjQ0hGNrdQgNQmhwHN8mbC7Ry+P4Olw/XkXQcRtqNDZAYxOKAMnY'
'tD6MDdDYlA5jQza2mo0N2NiMTgKisTlkwSELjllwzELUWQjIQlAWArMAcGwAx4ZwbAjHNiCGATEMjGFQDH0u38rX4frxKoS3GZtrtPpAdlDHjurYUR17jccO'
'eOwdQ0A+doE0ACA7umNHQnYt38vX4frx4r18m7C5hEEWDLJgmAXDLNTy2EEeO8ljR3nsUZOhAyA7ldUXMzbXyPqVSNgIKKsvJmwuMep9ADx2xGNHPEqNRwE8'
'ygxJEKSjtPooCLijNMqCoDxKXVgLFNaChbVgYS1ShyAQgsA7+TZhc4m6rhaQR0F5FJRHsToEYKMYnUZEo9RoFECjEBoF0Si1OgqQUUgdBcEoWUcA5ihJEaA4'
'ClTVAlW1YFUtWFVrTUYFMiqRUZGMWhfVCmBUKqoVuag1FxW4qJ0iQCxqXVQrOKNSUa2ojKr1/5OCM6rSn4OiNGrNRQUuKjqjIhgVqmqFqlqxqlasqrVuOSq0'
'HJVajootR62dUcEZlZxR0Rl11BEAGRXJqEhGq/uNBv1Go7sYw3ajgTMaoNHQGQ3ZaDUbDdhoxEZDNlqtjAbKaIJJQGe02hkNnNHQGQ2d0Wo2GrDRyBkN0WgO'
'pxGk0Rw3Aq3RajQaoNEIjYZotNoaDazRyBoNrdGg3WhQTxvW04b1tNfW6GCNTtboaI1eW6MDGh3R6IhGr5uNDs1Gp7sYx16jS30UHNjoyEZHNnrda3TwRqde'
'o6M2en0X43AX43QX43gX43U57UBGp3LaEYxel9MO5bRTOe1YTnvtjA7O6En/kY7S6FBPO1ijozU6WmPUZAwgY+BFTCAaA7QxgI2BbAxkY9TaGKCNga3GQG+M'
'2hsD2BjIxkA2Ru2NAd4Y9A1PoDaGwVkAOAZ+xBNIx/A6C1BSh1MWsKKOWhsDtDFIGwO1MbJGUwAeA/EYiMeoa+oAOsagLCAcs4ZjAhyTtDGRjVlrYwIaE9GY'
'iMas0ZiAxqSKOpGMCVfUCe3GxCvqxH5jwkc8CWxMrKkT4ZhwSZ1QVCc2HBOr6oSGYwIdExuOiXhMuKZOsMfEa+pEfcy6rk6oq5Pq6sS6OsEeE/iYaI+JgBx1'
'z3FAz3FQz3Fgz3HU338P+IhnNED0wG94Rs3HAXwcxMfxHh9/XU1P+9OXdbqZnv2L3Z7X/el5Pf+4fXlw+PR7xunbeb/ePp7u198LT4/r12V3PC6Py/Xf//3u'
'5EPb5cfp178k51W1'
)))
assert hashlib.sha256(V66_BLOB_2).hexdigest()=='9ea5851a0d40a5cedee399f4f950c068ddc03470c999803ef4945f80e1ed0a93'
V66_ASSETS['inputs/8b-reference.json']=V66_BLOB_2
V66_BLOB_3 = zlib.decompress(base64.b64decode((
'eNpdVkuOHEUQ3c8pQkJIIHW3xmPP2JblxRgZg4Rky4wlezfRVVHd6cnKLPLTH1aGFawQK8QVOIN33MQ+gK/Ai8yq7sGb/lRlZkS89+JFfkGbi8UF0Uu/zDFR71uxtJQ1b4wPFGRjZEsnJ89eXH149+f54pzoW7OTlqLjIa59IrpeDWmON/Oz07OL+em9+dnda6JXUXCoxMG7KJEuX3xP9DyYlXFsqcuuScY7St7bGfXSmtwTXa2FWlkFEfIdYnP0zrjVDA87zjZR5H6w5WWUsDENDmaHXBI+ObTTU7LGSVxQjfrSZ6x5NRDdO32ISp87nJDTkJF78jfiPr3/jYxrbG4R7Bj26xltTVqTx/qQHQ0SqPGuNZr5gi67hAdaAMlOmqxPZ5RQQmONuISDUg4ujpDWiHF2K5K4JuyHBDAPMckk6bFIqypH+RAKhmXDSlxGZTUongOSiPN0pQnkJ3R9aCXMKEfdQ9cx+SCPO7ZRrj+++xV0g0RSJBTvrQ+2PcRLHFaSiANKlkFYk1vudVFba/NYDECA7YMndMhB13tn94pPCt5abFPsjJvHRhwH4/GqH/ADdYKZZ+IksDU/c5EBtjWiWGmILltbQZvRELArjXig6rlxQL3jRqFxnVnlwBV4BynifO5FaYkNW8F2H9M8BTZVRhA0OwDWmo1pM5DilMCUZrAWbhVK5Ge5EnxyQk+5WY8IrRnZbT1K/ilLTKjhNVoBJxaaMmgC5SvPSPoNoYGsdyvs8BU1fIRp64KuON7gH8htIzVrdqsRviI1qQlQZ0TB7m6doOqgr4axKz+9/4NeBOkEwduKAMeY0U1E6NfCL4jpta+XMtI2Kv53KKHwn9babuigkFQugwfClVAlzAJN4py88/1e6RAQig2KJw57/fHdL5rC8SHRGz35CfLVZMzKIZvOBKQQkww4uUlAHrXWppFSHfpac/DNTdW20s0D9AfCouoZug57WhvVsmluOwj+2FgbdWqPUZWPaKlZlI4qBfGGjeWlBQxaee3LKDARJ6bgKxu2uaKPhWJjKTmiI6bGcHErISrYSI22ASzPrfc3eaAthwqLHt750CNN4LoqUk3QYCxd9+9fdKYfd+kxnd9HF0Kfb6XRwgQ5o+2syntqNgt1ZD2kpjuSohnKrrhhtcDR0OIekXYVDodCNnJEKg7SmA7o6T+UsF0beOmNyKC8JdXk4K1pwHMhAbknGb0oQqu6CpVEkACt1zKj2c2VVzhub9LobTL2cZkTR+7rjtK2wKakhGYFnaM1aKxqGkdNFBZm4yGKsyYBrLXHfpAY6xtVItGHv/8pESxjrS82Scu8L891/cFVAYVYIF7CDFMDAfk2V6TGBltUl7ykycNgtkAbu9VoIN2BTRiHBGSEXqb7F3dOz3Tuzcuo0mzHBt6uPdA+nkR3T09PiX4UHSkzWilVKFgtJpq2TK+H51+SCpcrcvpLE+g53CBdNIbOSm7Va7UoxR/pwTwZdKSEcEuTHpHpEOzBPbptzrc1B2tDbOeVPR03AKZMEIgRM0Mdb/ChggVU55pdCw50HC7LaO20F0CPTMB/dhB6PnjlyudwO4llYNeskQAg4gM01MHzjwpYlUmBTEc8MMeTH4o7X1Zf8hlIcOIl48oB8+/LEDyQ1Hibe4diynGHuae9dWBjHGeHPv18lmVn1E9KGfP/DR5qTQf9YJRLNZlYCFW3YqSxj8rky2L1NIFy7GEoG3wtGb5XmriYUAVfLWaa5Ni1luamzLADtvMRW4wwjMioq7oc2U63hwpEybFeL5DIlez0XjJdyjAWkylmrKPJ6Kxgp3Ox6h8FW8AzWUY5dgyC7551XNZene6LeNEKKNHAPizo6Q490ECY08UMneZDRONkcHW4No0n6+iq7ENbxWb0aSMG5RHgVnASpiUAzK5MK46fX1hrgAV9Mw0vqlc+1Ii7BK6WsTRx1bRqUKDecQgd1YAWpsFYvd0e7kjQyB08v8ScbNBaFHtMUtLDZFdcr625j85f2TfVnRUrVZSyqLCPd5j6ElcC06vVoi71MlyaMfiwFkC1vox1RWsK+J3fKk37CUsutLXF7rh9q8M+HQeQytM0GIdw9WKmegmaKl3Qfxm4Ufk='
)))
assert hashlib.sha256(V66_BLOB_3).hexdigest()=='ede6a8454c009aa95bf4dc322029832c62f8df5fae5d57e901e367f2caabf241'
V66_ASSETS['inputs/protocol.md']=V66_BLOB_3


In [ ]:
#@title  Initialization and offline input check 
V66_NS={"__name__":"v66_study"}
V66_SOURCES=[V66_SHARED,V66_NATIVE66,V66_NATIVE66_REPORTS]
V66_HASHES=[hashlib.sha256(src.encode()).hexdigest() for src in V66_SOURCES]
for i,src in enumerate(V66_SOURCES):exec(compile(src,f"<v66-{i}>","exec"),V66_NS)
V66_INPUT,V66_SMOKE,V66_REF=[json.loads(V66_ASSETS[k]) for k in ("inputs/formal.json","inputs/smoke.json","inputs/8b-reference.json")]
V66_NS["audit66"](V66_INPUT,V66_SMOKE,V66_REF)
print(" Checked  96  Its an official scenarios. 2   smoke  scenarios and  384  A historical comparison. This level is not called  API。")


In [ ]:
#@title  from  Colab Secrets  Initialization  API  Client 
from google.colab import userdata
def create_v66_client():
    hint=" Please.  Colab Secrets  Settings  OPENAI_API_KEY  And turn this on.  notebook  Access rights. "
    try: key=userdata.get("OPENAI_API_KEY")
    except Exception: raise RuntimeError(hint) from None
    if not isinstance(key,str) or not key.strip():raise RuntimeError(hint)
    return V66_NS["OpenAI66"](key.strip())
V66_STUDY=V66_NS["Study66"](V66_RUN/"experiment",V66_INPUT,V66_SMOKE,V66_REF,V66_ASSETS,V66_HASHES,
                             create_v66_client(),USD_BUDGET,RETRY_UNCERTAIN)
print(" Existing complete trajectory ",V66_STUDY.completed("smoke"),"/12 smoke；",V66_STUDY.completed("full"),"/576  Formal. ")


In [ ]:
#@title  View  v6.6   API  Reasons for refusal (no request sent) 
import re
import sys


def v66_http_detail(response, *args, **kwargs):
    if response.status_code not in (400, 401, 403, 404, 422, 429):
        return
    try:
        error = response.json()["error"]
    except (ValueError, KeyError, TypeError):
        error = {}
    if not isinstance(error, dict):
        error = {}
    # Authentication errors sometimes echo part of the supplied credential.
    fields = ("type", "code", "param") if response.status_code == 401 else ("type", "code", "param", "message")
    parts = [f"{field}={error[field]}" for field in fields if isinstance(error.get(field), str) and error[field]]
    detail = " | ".join(parts) or " The server did not provide a visible error field. "
    sent = response.request.headers.get("Authorization", "").removeprefix("Bearer ")
    if sent:
        detail = detail.replace(sent, "[ Hide ]")
    detail = re.sub(r"(?i)\b(?:bearer\s+\S+|(?:sk-|sess-|hf_)[\w.*-]+)", "[ Hide ]", detail)
    print(f"OpenAI HTTP {response.status_code} | {' '.join(detail.split())[:600]}", flush=True)
    if len(sent.strip()) <= 1:
        print(" Last requested key is empty or only one character. Please update  OPENAI_API_KEY， Runs the client initialization again. ")


def v66_diagnose():
    session = V66_STUDY.provider.session
    # Rerunning this cell replaces only our hook. requests expects it to return None.
    hooks = session.hooks["response"]
    hooks[:] = [hook for hook in hooks if not getattr(hook, "_v66_http_detail", False)]
    v66_http_detail._v66_http_detail = True
    hooks.append(v66_http_detail)
    tb = getattr(sys, "last_traceback", None) or getattr(getattr(sys, "last_exc", None), "__traceback__", None)
    while tb is not None:
        frame = tb.tb_frame
        if frame.f_code.co_name == "__call__" and frame.f_locals.get("self") is V66_STUDY.provider:
            response = frame.f_locals.get("response")
            if response is not None and response.status_code in (400, 401, 403, 404, 422, 429):
                v66_http_detail(response)
                return
        tb = tb.tb_next
    print(" Last one is not kept at the moment  API  rejection details. If the next model call is rejected, its reason is displayed automatically. ")


v66_diagnose()


##  Call Model 

 Local budgets based on official standard rates, API  returned usage and a conservative reservation for the next request control new calls; the actual bill follows  API  The platform will be subject to your input, cache hits, and reasoning.  token  The amount varies depending on the behavior of the model.  API  turns receive separate receipts, and saved receipts are reused after ordinary disconnections. If interruption occurs after sending a request but before persisting its receipt, flag that call as having unknown cost. After checking usage, explicitly enable  `RETRY_UNCERTAIN`  Retry; the conservative reservation to the old request is retained. 

 During the formal stage, run all six conditions for every scenario with the same model configuration across stages. Only in an independent  smoke  The original tool is confirmed on the upper level that the original tool is being returned and used in normal quantities. 


In [ ]:
#@title  Start or continue the current phase (call fee)  API）
V66_RESULT=None
V66_STUDY.run(RUN_STAGE)


In [ ]:
#@title  No Logic Export and Results Show 
V66_RESULT=None
V66_RESULT=V66_NS["export66"](V66_STUDY,RUN_STAGE)
rep=V66_RESULT[0]
print(" At this stage  API  Usage ",rep["meter_stage"],flush=True)
if RUN_STAGE=="smoke" and rep["status"]=="COMPLETE":
    m=rep["meter_stage"]
    native=sum(sum(e["origin"]=="model" and bool(e["action"]) for e in s["result"]["events"])
               for k,s in V66_STUDY.states.items() if k.startswith("smoke/") and s.get("result"))
    print("smoke  The model emits a deconstructable primary tool to call ",native,
          "； Press  2  Exploiting of a scenarios. Rough estimates at the formal stage.  USD",round(m["estimated_usd"]*48,2),
          "（ Add  smoke  Costs have been incurred and actual usage may vary). ",flush=True)
elif rep["primary"] is not None:
    print(" Main indicator (percentage points) ",rep["primary"])
    print(" and  8B  The difference between the main aspects of the same scenarios (secondary) ",next(e for e in rep["effects"] if e["contrast"]=="strong_minus_8B"))
    print(" In case of zero misset, correspond to one side of the top ( %）",rep["zero_natural_any_y_upper95_world_pct"]," Autonomy  /",rep["zero_controlled_any_y_upper95_world_pct"]," Controlled ")
for g in rep["conditions"]:
    print(g["context"],g["start"]," Total ",g["n"]," The right goods. ",g["final_x"]," The other sides merchandise. ",g["final_y"],
          " Stand-by. ",g["fallback"]," Other ",g["other_order"]," Unlisted ",g["no_order"]," Response pending review ",g["text_unreviewed"])


##  Fetch Results 

 The official package of the default labels is on  `MyDrive/AAAI_Workshop/agent_entrainment/agent_entrainment_v6p6_native_tools/native-responses-medium/v6p6_native_behavior96_full_COMPLETE_results.zip`。 Retain final orders, termination reasons, native calls, and cost receipts for every condition. When a plain-text response occurs, `full-text-review.csv`  produces a text-review queue with shuffled identifiers and item identifiers removed. To distinguish clarification requests from other text, fill in  `label`（`clarification`、`decline`  or  `other`）， Can not open message  `text-decisions.csv`， Rerun the export cell without additional inference charges. Unannotated responses remain pending review. Formal completion  ZIP  Automatically disconnects after readback and synchronisation has been successful. 


In [ ]:
from google.colab import drive,runtime
V66_NS["finish66"](V66_STUDY,V66_RESULT,V66_MOUNT,AUTO_DISCONNECT,os.path.ismount,drive.flush_and_unmount,runtime.unassign)
